# Fine-tune PhoBERT 3 nhãn cho web sqlite_realtime

Notebook chạy trực tiếp trên máy local, giữ đúng contract 3 nhãn của web: `khác`, `ủng hộ`, `phân biệt`. Toàn bộ dữ liệu cũ được chuẩn hóa, loại trùng và loại nhóm nhãn mâu thuẫn rồi chia lại leakage-safe 70/15/15; toàn bộ dữ liệu mới hợp lệ được thêm vào train. Model cuối được lưu vào `artifacts/phobert`, đúng đường dẫn mà `.env` của web đang dùng.

## 1. Cài thư viện
Chạy cell dưới nếu môi trường Python hiện tại còn thiếu thư viện. Máy này có NVIDIA RTX 5060 Ti 16 GB nên cấu hình mặc định dùng GPU khi PyTorch nhận CUDA.

In [ ]:
%pip install -q pandas scikit-learn torch transformers accelerate underthesea

## 2. Cấu hình đường dẫn local
Hãy khởi động Jupyter trong `D:\Kafka_Web_Bac_Nam\sqlite_realtime`. Notebook tự đọc dữ liệu cũ và dữ liệu mới từ thư mục cha `D:\Kafka_Web_Bac_Nam\data`, lưu checkpoint theo từng lần chạy trong `artifacts/training_runs`, và chỉ ghi model cuối vào `artifacts/phobert`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json, sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'src' / 'virdc_stream').exists():
    raise FileNotFoundError('Hãy mở Jupyter tại D:/Kafka_Web_Bac_Nam/sqlite_realtime rồi chạy lại notebook.')
SOURCE_PROJECT_DIR = PROJECT_DIR.parent
RUN_NAME = datetime.now(timezone.utc).strftime('phobert_3class_%Y%m%d_%H%M%S')
RUN_DIR = PROJECT_DIR / 'artifacts' / 'training_runs' / RUN_NAME
MODEL_DIR = PROJECT_DIR / 'artifacts' / 'phobert'
PREPARED_DIR = RUN_DIR / 'prepared_data'

OLD_DATA_DIR = SOURCE_PROJECT_DIR / 'data' / 'processed'
NEW_DATA_DIR = SOURCE_PROJECT_DIR / 'data' / 'add4000' / 'processed'
for required in [OLD_DATA_DIR, NEW_DATA_DIR, PROJECT_DIR / 'src' / 'virdc_stream' / 'merged_training_data.py']:
    if not required.exists():
        raise FileNotFoundError(f'Không tìm thấy dữ liệu hoặc code bắt buộc: {required}')
RUN_DIR.mkdir(parents=True, exist_ok=False)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_DIR.mkdir(parents=True)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('Run directory:', RUN_DIR)
print('Web model directory:', MODEL_DIR)

Run directory: D:\Kafka_Web_Bac_Nam\sqlite_realtime\artifacts\training_runs\phobert_3class_20261110_155203
Web model directory: D:\Kafka_Web_Bac_Nam\sqlite_realtime\artifacts\phobert


## 3. Gộp dữ liệu 3 nhãn và loại trùng
Sau khi chia dữ liệu, giữ toàn bộ câu gốc và chỉ thêm biến thể không dấu vào train. Bỏ qua biến thể trùng, có nhãn mâu thuẫn hoặc có cùng dạng không dấu với câu ở validation/test.
Dữ liệu cũ giữ nguyên `khác`, `ủng hộ`, `phân biệt`, nhưng split cũ không được giữ vì có rò rỉ lớn. Sau full dedup, dữ liệu cũ được chia lại 70/15/15 theo nhãn. Dữ liệu mới có `HOSTILE` hoặc `PREJUDICIAL` ở bất kỳ vùng nào được ánh xạ thành `phân biệt`; `NON_HATE/NON_HATE` được ánh xạ thành `khác` và chỉ được thêm vào train. Dataset mới không tạo thêm mẫu `ủng hộ`.

In [2]:
import unicodedata

import pandas as pd
from IPython.display import display
from virdc_stream.merged_training_data import (
    augment_train_with_accentless_variants,
    prepare_merged_three_class,
)
from virdc_stream.preprocessing import normalize_text


def accentless_sentence(text):
    decomposed = unicodedata.normalize('NFD', text)
    without_marks = ''.join(char for char in decomposed if not unicodedata.combining(char))
    return normalize_text(without_marks.replace('đ', 'd').replace('Đ', 'D'))

splits, data_report = prepare_merged_three_class(OLD_DATA_DIR, NEW_DATA_DIR)
splits, augmentation = augment_train_with_accentless_variants(splits)
EXPECTED_ROWS = {'train': 9301, 'val': 1071, 'test': 1071}
assert data_report['final_rows'] == EXPECTED_ROWS, data_report['final_rows']
assert data_report['old_source_rows'] == 17390
assert data_report['old_unique_nonconflicting_rows'] == 7137
assert not any(data_report['final_cross_split_unique_text_overlap'].values())
assert data_report['new_source_rows'] == 4423
assert data_report['new_rows_retained'] == 4306
assert data_report['new_duplicate_rows_removed'] == 62
assert data_report['new_conflicting_duplicate_text_groups'] == 6
assert data_report['new_conflicting_duplicate_rows_removed'] == 32
assert data_report['new_rows_removed_for_old_overlap'] == 23

original_train = splits['train']
held_out_sentences = {
    accentless_sentence(text)
    for name in ('val', 'test')
    for text in splits[name]['sentence']
}
variants = original_train.assign(sentence=original_train['sentence'].map(accentless_sentence))
label_counts = variants.groupby('sentence')['label'].transform('nunique')
variants = variants.loc[
    variants['sentence'].ne('')
    & ~variants['sentence'].isin(original_train['sentence'])
    & ~variants['sentence'].isin(held_out_sentences)
    & label_counts.eq(1)
].drop_duplicates(subset='sentence')
splits['train'] = pd.concat([original_train, variants], ignore_index=True)
data_report['train_augmentation'] = {
    'method': 'accentless-variants',
    'original_rows': len(original_train),
    'added_rows': len(variants),
}
data_report['final_rows']['train'] = len(splits['train'])
data_report['final_label_distribution']['train'] = splits['train']['label'].value_counts().to_dict()

for split, frame in splits.items():
    frame.to_csv(PREPARED_DIR / f'{split}.csv', index=False)
(RUN_DIR / 'data_report.json').write_text(
    json.dumps(data_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(pd.DataFrame(data_report['final_label_distribution']).fillna(0).astype(int))
display(pd.Series({
    'Dữ liệu cũ ban đầu': data_report['old_source_rows'],
    'Dòng cũ thuộc nhóm nhãn mâu thuẫn bị loại': data_report['old_conflicting_rows_removed'],
    'Dòng cũ trùng cùng nhãn bị loại': data_report['old_duplicate_rows_removed'],
    'Dữ liệu cũ duy nhất còn lại': data_report['old_unique_nonconflicting_rows'],
    'Dữ liệu mới ban đầu': data_report['new_source_rows'],
    'Dòng trùng cùng nhãn bị loại': data_report['new_duplicate_rows_removed'],
    'Dòng thuộc nhóm nhãn mâu thuẫn bị loại': data_report['new_conflicting_duplicate_rows_removed'],
    'Câu trùng dữ liệu cũ bị loại': data_report['new_rows_removed_for_old_overlap'],
    'Mẫu mới sạch giữ lại': data_report['new_rows_retained'],
    'Train gốc': len(original_train),
    'Biến thể không dấu được thêm': len(variants),
    'Train cuối': len(splits['train']),
    'Validation sau chia lại': len(splits['val']),
    'Test sau chia lại': len(splits['test']),
}, name='Số lượng'))

,train,val,test
khác,7033,769,769
phân biệt,1981,241,241
ủng hộ,287,61,61


Dữ liệu cũ ban đầu                           17390
Dòng cũ thuộc nhóm nhãn mâu thuẫn bị loại     2621
Dòng cũ trùng cùng nhãn bị loại               7632
Dữ liệu cũ duy nhất còn lại                   7137
Dữ liệu mới ban đầu                           4423
Dòng trùng cùng nhãn bị loại                    62
Dòng thuộc nhóm nhãn mâu thuẫn bị loại          32
Câu trùng dữ liệu cũ bị loại                    23
Mẫu mới sạch giữ lại                          4306
Train cuối                                    9301
Validation sau chia lại                       1071
Test sau chia lại                             1071
Name: Số lượng, dtype: int64

## 4. Chuẩn bị PhoBERT theo đúng pipeline của web
Notebook dùng trực tiếp `normalize_text`, `segment_text` và thứ tự nhãn từ source của `sqlite_realtime`, bảo đảm model lưu ra được `ModelService` chấp nhận.

In [3]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
)
from virdc_stream.phobert import LABELS as WEB_LABELS, segment_text
from virdc_stream.preprocessing import PREPROCESSING_VERSION

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

LABELS = ('khác', 'ủng hộ', 'phân biệt')
assert LABELS == tuple(WEB_LABELS)
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
ID_TO_LABEL = {index: label for label, index in LABEL_TO_ID.items()}
BASE_MODEL = 'vinai/phobert-base-v2'
SEED = 42
EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 5e-6
MAX_LENGTH = 256
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

class ThreeClassTextDataset(Dataset):
    def __init__(self, frame):
        segmented = [segment_text(text) for text in frame['sentence']]
        self.encodings = tokenizer(segmented, truncation=True, max_length=MAX_LENGTH)
        self.labels = [LABEL_TO_ID[label] for label in frame['label']]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: values[index] for key, values in self.encodings.items()}, 'labels': self.labels[index]}

datasets = {name: ThreeClassTextDataset(frame) for name, frame in splits.items()}
train_label_ids = np.asarray([LABEL_TO_ID[label] for label in splits['train']['label']])
CLASS_WEIGHTS = torch.tensor(
    compute_class_weight('balanced', classes=np.arange(len(LABELS)), y=train_label_ids),
    dtype=torch.float32,
)
print({name: len(ds) for name, ds in datasets.items()})
print('Class weights:', dict(zip(LABELS, CLASS_WEIGHTS.tolist())))

Training device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti
{'train': 9301, 'val': 1071, 'test': 1071}
Class weights: {'khác': 0.44082656502723694, 'ủng hộ': 10.802555084228516, 'phân biệt': 1.5650345087051392}


## 5. Fine-tune tối đa 10 epoch với early stopping
Model được đánh giá sau mỗi epoch. Nếu validation Macro-F1 không cải thiện trong 2 epoch liên tiếp, quá trình train dừng sớm và khôi phục checkpoint tốt nhất.

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID
)

def compute_metrics(eval_prediction):
    predictions = np.argmax(eval_prediction.predictions, axis=-1)
    actual = eval_prediction.label_ids
    return {
        'accuracy': accuracy_score(actual, predictions),
        'macro_f1': f1_score(actual, predictions, labels=[0, 1, 2], average='macro', zero_division=0),
    }

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=str(RUN_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    seed=SEED,
    report_to='none',
)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['val'],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)],
    class_weights=CLASS_WEIGHTS,
)
train_result = trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.786000,0.909772,0.784314,0.616622
2,0.639800,0.696364,0.800187,0.696405
3,0.609700,0.749910,0.826331,0.730473
4,0.580500,0.782857,0.834734,0.747701
5,0.525300,0.898302,0.837535,0.741204
6,0.508900,0.980177,0.822596,0.720947


## 6. Đánh giá và lưu model lên Drive

In [5]:
def evaluate_split(name):
    output = trainer.predict(datasets[name])
    predicted = np.argmax(output.predictions, axis=-1)
    actual = output.label_ids
    return {
        'rows': int(len(actual)),
        'accuracy': float(accuracy_score(actual, predicted)),
        'macro_f1': float(f1_score(actual, predicted, labels=[0, 1, 2], average='macro', zero_division=0)),
        'labels': list(LABELS),
        'classification_report': classification_report(
            actual, predicted, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0
        ),
        'confusion_matrix': confusion_matrix(actual, predicted, labels=[0, 1, 2]).tolist(),
    }

metrics = {
    'validation': evaluate_split('val'),
    'test': evaluate_split('test'),
    'best_validation_macro_f1': float(trainer.state.best_metric),
    'best_checkpoint': trainer.state.best_model_checkpoint,
}
metadata = {
    'model_version': RUN_NAME,
    'base_model': BASE_MODEL,
    'positive_label': 'phân biệt',
    'known_labels': list(LABELS),
    'preprocessing_version': PREPROCESSING_VERSION,
    'segmentation': 'underthesea-word_tokenize-text',
    'max_length': MAX_LENGTH,
    'seed': SEED,
    'max_epochs': EPOCHS,
    'completed_epochs': float(trainer.state.epoch),
    'early_stopping_patience': 2,
    'class_weights': dict(zip(LABELS, [float(value) for value in CLASS_WEIGHTS])),
    'train_rows': len(splits['train']),
    'train_augmentation': data_report['train_augmentation'],
    'validation_rows': len(splits['val']),
    'test_rows': len(splits['test']),
    'saved_at_utc': datetime.now(timezone.utc).isoformat(),
}

model = trainer.model
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
(MODEL_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
trainer.state.save_to_json(str(RUN_DIR / 'trainer_state.json'))

display(pd.DataFrame({name: {k: result[k] for k in ('rows', 'accuracy', 'macro_f1')}
                      for name, result in [('validation', metrics['validation']), ('test', metrics['test'])]}).T)
display(pd.DataFrame(metrics['test']['classification_report']).T)
display(pd.DataFrame(metrics['test']['confusion_matrix'], index=LABELS, columns=LABELS))

print('Đã lưu model web tại:', MODEL_DIR)
print('Metrics và trainer state tại:', RUN_DIR)

,rows,accuracy,macro_f1
validation,1071.0,0.834734,0.747701
test,1071.0,0.784314,0.674062


,precision,recall,f1-score,support
khác,0.862719,0.833550,0.847884,769.000000
ủng hộ,0.485294,0.540984,0.511628,61.000000
phân biệt,0.638462,0.688797,0.662675,241.000000
accuracy,0.784314,0.784314,0.784314,0.784314
macro avg,0.662158,0.687777,0.674062,1071.000000
weighted avg,0.790759,0.784314,0.787055,1071.000000


,khác,ủng hộ,phân biệt
khác,641,34,94
ủng hộ,28,33,0
phân biệt,74,1,166


Đã lưu model web tại: D:\Kafka_Web_Bac_Nam\sqlite_realtime\artifacts\phobert
Metrics và trainer state tại: D:\Kafka_Web_Bac_Nam\sqlite_realtime\artifacts\training_runs\phobert_3class_20261110_155203


## 7. Nạp lại model local và kiểm tra tương thích web
Cell này nạp lại model từ `artifacts/phobert`, kiểm tra metadata 3 nhãn và chạy dự đoán thử trước khi khởi động web.

In [6]:
saved_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
saved_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
examples = [
    'người miền nào cũng cần được tôn trọng',
    'đừng tuyển người vùng đó vào làm',
]
segmented = [segment_text(text) for text in examples]
batch = saved_tokenizer(segmented, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    probabilities = torch.softmax(saved_model(**batch).logits, dim=-1).cpu().numpy()
for text, probs in zip(examples, probabilities):
    predicted_id = int(probs.argmax())
    print({'text': text, 'label': saved_model.config.id2label[predicted_id], 'confidence': float(probs[predicted_id])})

{'text': 'người miền nào cũng cần được tôn trọng', 'label': 'khác', 'confidence': 0.8851602673530579}
{'text': 'đừng tuyển người vùng đó vào làm', 'label': 'khác', 'confidence': 0.8781255483627319}


In [ ]:
# In và lưu các mẫu test dự đoán sai
test_output = trainer.predict(datasets["test"])

logits = test_output.predictions
actual_ids = test_output.label_ids
predicted_ids = np.argmax(logits, axis=1)

probabilities = torch.softmax(
    torch.tensor(logits), dim=1
).numpy()

wrong_mask = predicted_ids != actual_ids
wrong_indices = np.flatnonzero(wrong_mask)

wrong_df = splits["test"].iloc[wrong_indices].copy().reset_index(drop=True)

wrong_df["actual_label"] = [
    LABELS[index] for index in actual_ids[wrong_indices]
]
wrong_df["predicted_label"] = [
    LABELS[index] for index in predicted_ids[wrong_indices]
]
wrong_df["prediction_confidence"] = probabilities[
    wrong_indices, predicted_ids[wrong_indices]
]

# Xác suất của từng nhãn
for label_id, label in enumerate(LABELS):
    wrong_df[f"prob_{label}"] = probabilities[wrong_indices, label_id]

wrong_df = wrong_df[
    [
        "sentence",
        "actual_label",
        "predicted_label",
        "prediction_confidence",
        "prob_khác",
        "prob_ủng hộ",
        "prob_phân biệt",
    ]
].sort_values("prediction_confidence", ascending=False)

total_errors = len(wrong_df)
error_rate = total_errors / len(splits["test"])

print(f"Tổng mẫu test: {len(splits['test'])}")
print(f"Dự đoán sai: {total_errors}")
print(f"Tỷ lệ sai: {error_rate:.2%}")

# Đổi thành None nếu muốn hiển thị toàn bộ
MAX_ERRORS = None
display(wrong_df.head(MAX_ERRORS))

# Lưu toàn bộ dự đoán sai
error_path = RUN_DIR / "test_misclassifications.csv"
wrong_df.to_csv(error_path, index=False, encoding="utf-8-sig")

print("Đã lưu tại:", error_path)

Tổng mẫu test: 1071
Dự đoán sai: 231
Tỷ lệ sai: 21.57%


,sentence,actual_label,predicted_label,prediction_confidence,prob_khác,prob_ủng hộ,prob_phân biệt
16,trò trẻ cá_nhân việt_nam đất hình_thành hy_sinh,khác,ủng hộ,0.988296,0.006870,0.988296,0.004834
105,trung chào anh_em nam bắc,khác,ủng hộ,0.987825,0.006302,0.987825,0.005874
181,ta bắc_nam miền trung kiểu,khác,ủng hộ,0.987783,0.006204,0.987783,0.006013
6,dân_tộc việt_nam chống giặc bắc trung nam anh_...,khác,ủng hộ,0.987782,0.006252,0.987782,0.005966
162,việt_nam ta dân_tộc ta,khác,ủng hộ,0.987755,0.006552,0.987755,0.005693
...,...,...,...,...,...,...,...
166,trumpky miền trung phân_biệt nam bắc,khác,ủng hộ,0.936289,0.036732,0.936289,0.026979
46,monkey không,khác,phân biệt,0.935240,0.060878,0.003882,0.935240
104,that nam cung eo ua bac va bac cung vay gio cu...,phân biệt,khác,0.935018,0.935018,0.006706,0.058275
118,ngoại miền bắc giỏi trí tiến_thủ,ủng hộ,khác,0.932601,0.932601,0.020604,0.046796


Đã lưu tại: D:\Kafka_Web_Bac_Nam\sqlite_realtime\artifacts\training_runs\phobert_3class_20261110_155203\test_misclassifications.csv
